# LLaVA 1.5 — Deletion Faithfulness Experiment

Clean per-token deletion experiment for `llava-hf/llava-1.5-7b-hf`.

In [ ]:
%pip install --quiet \
    "transformers>=4.40" \
    "accelerate>=0.28" \
    "bitsandbytes>=0.43" \
    "datasets" \
    "pillow" \
    "tqdm" \
    "huggingface_hub"
print("Installation complete.")

In [ ]:
import os
from huggingface_hub import login

# Set HF_TOKEN in your shell before launching: export HF_TOKEN=hf_...
token = os.environ.get('HF_TOKEN')
if token:
    login(token=token)
else:
    login()  # will prompt interactively

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_NVFUSER_DISABLE'] = '1'
os.environ['TORCH_NVFUSER_DISABLE'] = '1'
os.environ['PYTORCH_JIT_USE_NNC_NOT_NVFUSER'] = '1'
print('[env] CUDA_LAUNCH_BLOCKING=1, NVFuser disabled')

In [ ]:
import sys
from pathlib import Path

# Assumes the notebook lives inside the llava_med_llava1.5/ directory.
# If running from a different working directory, set MODULE_DIR explicitly.
MODULE_DIR = Path.cwd()
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
print(f'Module path: {MODULE_DIR}')

In [ ]:
from config import Config
from pathlib import Path

USE_COCO = True

cfg = Config()
cfg.model_id            = 'llava-hf/llava-1.5-7b-hf'
cfg.load_in_4bit        = True
cfg.attn_implementation = 'eager'       # required for output_attentions

if USE_COCO:
    cfg.dataset_name   = 'lmms-lab/COCO-Caption'
    cfg.dataset_split  = 'val'
    cfg.image_column   = 'image'
    cfg.caption_column = 'answer'       # list[str]; loader uses first caption
    cfg.prompt         = 'Generate a caption for this image.'
    cfg.output_dir     = Path('llava_output')
else:
    cfg.dataset_name   = 'eltorio/ROCOv2-radiology'
    cfg.dataset_split  = 'train'
    cfg.image_column   = 'image'
    cfg.caption_column = 'caption'
    cfg.prompt         = 'Write a single-sentence radiology caption for this medical image.'
    cfg.output_dir     = Path('llava_output')

cfg.num_samples       = 2
cfg.max_new_tokens    = 100
cfg.attention_layer_strategy = 'global'

cfg.methods             = ['attention', 'gradcam' ,'gmar_l1', 'gmar_l2']
cfg.mask_ratios         = [0.1, 0.2, 0.3, 0.4, 0.5]
cfg.save_visualizations = True

EVAL_MODE    = 'per_token'
CONTENT_ONLY = False

cfg.output_dir.mkdir(parents=True, exist_ok=True)

[config] Model   : llava-hf/llava-1.5-7b-hf
[config] Dataset : ROCO v2
[config] Output  : results_deletion_roco
[config] Methods : ['attention', 'gmar_l2', 'gmar_l1', 'gradcam']


In [ ]:
from model_utils import load_model_and_processor
model, processor = load_model_and_processor(cfg)

In [ ]:
from dataset import load_dataset_samples
samples = load_dataset_samples(cfg)

In [ ]:
import gc
import json
import time
import torch
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm as tqdm_nb

from model_utils import (
    generate_caption, get_tokenizer, get_image_token_positions,
    get_token_probabilities, get_content_token_mask, build_tf_inputs,
)
from saliency import get_saliency_fn
from evaluation import evaluate_faithfulness_per_token, evaluate_faithfulness_random
from visualization import (
    save_token_saliency_grid, save_comparison_figure, save_perturbation_curve,
)

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / 'config.json', 'w') as f:
    json.dump(vars(cfg), f, indent=2, default=str)

tok = get_tokenizer(processor)
all_sample_results = []
t_total = time.time()

for i in tqdm_nb(range(len(samples)), desc='Samples'):
    sample      = samples[i]
    image       = sample['image']
    ref_caption = sample.get('caption', '')
    sample_id   = sample.get('id', str(i))

    # 1. Generate caption
    gen_ids, gen_text, input_len, inputs = generate_caption(model, processor, image, cfg)
    total_len = gen_ids.shape[1]
    num_gen   = total_len - input_len
    print(f'\nSample {i}: {num_gen} tokens — {gen_text[:80]}')
    if num_gen == 0:
        continue

    # 2. Setup
    img_positions     = get_image_token_positions(inputs, cfg.image_token_index)
    tf_inputs         = build_tf_inputs(inputs, gen_ids, input_len)
    orig_probs        = get_token_probabilities(model, tf_inputs, gen_ids, input_len)
    token_strings     = {
        pos: tok.decode([gen_ids[0, pos].item()], skip_special_tokens=True).strip()
        for pos in range(input_len, total_len)
    }
    content_mask      = get_content_token_mask(tok, gen_ids, input_len)
    eval_content_mask = list(content_mask) if CONTENT_ONLY else None
    content_positions = [
        pos for pos, keep in zip(range(input_len, total_len), content_mask) if keep
    ]

    sample_dir = out_dir / f'sample_{i:04d}'
    if cfg.save_visualizations:
        sample_dir.mkdir(parents=True, exist_ok=True)

    # 3. Per-method saliency + deletion evaluation
    method_results = {}
    all_saliency   = {}
    for method in cfg.methods:
        sal_maps = get_saliency_fn(method)(model, tf_inputs, gen_ids, input_len, img_positions, cfg)
        all_saliency[method] = sal_maps
        ev = evaluate_faithfulness_per_token(
            model, inputs, gen_ids, input_len, sal_maps, orig_probs, cfg,
            content_mask=eval_content_mask,
        )
        for row in ev.get('per_token', []):
            row['token_text'] = token_strings.get(row.get('position'), '')
        print(f'  {method}: AOPC={ev["aopc"]:.4f}')
        method_results[method] = ev

        if cfg.save_visualizations:
            save_token_saliency_grid(
                image, sal_maps, token_strings, method,
                str(sample_dir / f'saliency_{method}.png'),
                content_positions=content_positions or None,
            )

    # 4. Random baseline
    random_positions = (
        [p for p, k in zip(range(input_len, total_len), content_mask) if k]
        if CONTENT_ONLY else list(range(input_len, total_len))
    )
    rand_ev = evaluate_faithfulness_random(
        model, inputs, gen_ids, input_len, orig_probs, cfg,
        content_mask=content_mask if CONTENT_ONLY else None,
        token_positions=random_positions,
    )
    for row in rand_ev.get('per_token', []):
        row['token_text'] = token_strings.get(row.get('position'), '')
    print(f'  random: AOPC={rand_ev["aopc"]:.4f}')
    method_results['random'] = rand_ev

    # 5. Visualizations
    if cfg.save_visualizations:
        if len(all_saliency) >= 2:
            save_comparison_figure(
                image, all_saliency, token_strings,
                str(sample_dir / 'comparison.png'),
                content_positions=content_positions or None,
            )
        save_perturbation_curve(
            method_results, str(sample_dir / 'perturbation_curve.png'),
            title=f'Sample {i}',
        )
        image.save(str(sample_dir / 'original.png'))

    all_sample_results.append({
        'sample_id':            sample_id,
        'sample_idx':           i,
        'generated_text':       gen_text,
        'reference_caption':    ref_caption,
        'num_generated_tokens': num_gen,
        'num_content_tokens':   sum(content_mask),
        'eval': {
            m: {
                'aopc':                r.get('aopc', 0.0),
                'mean_drops_by_ratio': r.get('mean_drops_by_ratio', {}),
                'per_token':           r.get('per_token', []),
            }
            for m, r in method_results.items()
        },
    })

    del tf_inputs, orig_probs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

elapsed = time.time() - t_total
print(f'\nDone: {len(all_sample_results)} samples in {elapsed:.0f}s')
print(f'Output: {out_dir}')


## Save Results
Flatten per-token rows and write `per_token_drops.csv` + `summary.csv`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

out_dir = Path(cfg.output_dir)

# ── Per-token drops CSV ────────────────────────────────────────────────────
rows = []
for res in all_sample_results:
    base = {'sample_idx': res['sample_idx'], 'sample_id': res['sample_id']}
    for method, ev in res.get('eval', {}).items():
        for r in ev.get('per_token', []):
            rows.append({**base, 'method': method, **r})

csv_path = out_dir / 'per_token_drops.csv'
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f'Saved {len(rows)} rows → {csv_path}')

# ── AOPC summary ───────────────────────────────────────────────────────────
all_eval = {}
for res in all_sample_results:
    for m, ev in res.get('eval', {}).items():
        all_eval.setdefault(m, []).append(ev.get('aopc', 0.0))

summary = pd.DataFrame([
    {'method': m, 'mean_aopc': np.mean(v), 'std_aopc': np.std(v), 'n_samples': len(v)}
    for m, v in all_eval.items()
]).sort_values('mean_aopc', ascending=False).reset_index(drop=True)

summary.to_csv(out_dir / 'summary.csv', index=False)
print(summary.to_string(index=False))